# Lab 2: Visualizing Transformer Dataflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab2_Transformer_Dataflow_STUDENT.ipynb)

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 2, Dataflow Execution Models  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU optional (this lab is static analysis)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Trace a transformer block into an FX graph with torch.fx.symbolic_trace.
2. Annotate every node with FLOPs and bytes and compute its arithmetic intensity.
3. Classify nodes as compute-bound / memory-bound against your device's true ridge.
4. Predict which nodes become scheduling bottlenecks on a dataflow accelerator.

## Prerequisites
Lab 1 complete, plus the Week 2 lecture on dataflow execution and FX graphs.

## How this lab is graded (50 points)

You run this lab here on **Google Colab**, then submit your results in a
**separate graded notebook on Coursera Labs**. Nothing you do in *this* notebook
is graded directly — the grade comes from the numbers you carry over.

| Step | Where | What you do |
|------|-------|-------------|
| 1 | GitHub → Colab | Open [Lab2_Transformer_Dataflow_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab2_Transformer_Dataflow_STUDENT.ipynb), then **File → Save a copy in Drive** |
| 2 | Colab | Complete every TODO and run all cells. **A GPU is optional for this lab** — it is static analysis |
| 3 | Colab | Run **Part 7**, the final cell. It writes **`lab2_results.json`** and prints it inline |
| 4 | Colab | Copy the JSON straight from that cell's output, or open the file browser (📁, left sidebar) and double-click `lab2_results.json` |
| 5 | **Coursera Labs** | Paste it into the `RESULTS_JSON` cell of the graded notebook, then run every cell top to bottom |

The graded notebook has **no GPU, no model download and no network access**,
so it cannot re-trace your graph. What it *can* do — and what it does — is check
that your numbers are **internally consistent**: the FLOP and byte totals have
to follow from your per-node table, each node's arithmetic intensity from its
own FLOPs and bytes, and both class partitions from those same numbers against
the ridges you used.

Lab 2 has an advantage over Lab 1 here. Lab 1 measured *time*, which varies with
hardware. Lab 2 does **static analysis**: a fixed architecture traced at fixed
dimensions produces a **deterministic** graph. The node counts, the shapes and
the FLOP totals are therefore *exact* — the same on a T4, an A100, or a
CPU-only runtime.

| Part | Points | What it checks |
|------|-------:|----------------|
| **A — Graph structure invariants** | 10 | The traced graph: op histogram, node/edge counts, the shapes you read off the figure |
| **B — Cost-model self-consistency** | 18 | Your totals, AI ranking and data movers recomputed from your own per-node FLOPs and bytes |
| **C — Ridge classification & the ridge shift** | 12 | Both partitions recomputed from your numbers; the classification must *flip* when the ridge changes |
| **Graded notebook total** | **40** | fully autograded from your JSON |
| Concept Check quiz | 10 | separate auto-graded Coursera item |
| **Lab total** | **50** | |

**40 points is a full score in the graded notebook, and a CPU-only Colab run
earns all of it.** Every check there is **autograded from the JSON you paste** —
there is no written analysis and no human review, so your score is final the
moment you run the cells. A **score table** near the end of that notebook tells
you your total before you submit.

> **No GPU is required for a full score.** Lab 2 is static analysis:
> `symbolic_trace` and `ShapeProp` produce the same graph, the same shapes and
> the same FLOP and byte totals on a T4 or on a CPU-only runtime. Every one of
> the 40 points is reachable either way.
>
> You are **encouraged** to re-run the lab on other GPUs — the one
> hardware-dependent number here is your device ridge, and watching the
> classification move as the ridge changes is the point of Part 4. It earns
> **no extra credit**; the full 40 points come from a single free-tier run.

> **There is also a separate Concept Check quiz** on Coursera, auto-graded as
> its own item and worth **10 points**. It is not part of the graded notebook —
> take it as well. Together the two make up the lab's 50 points.

> **Rounding is fine.** Every numeric check allows 2% relative tolerance, which
> is far more than the rounding in the lab's printed output. Transfer the values
> as the lab produced them and you will be well inside it.

The **Checkpoint** cells in this notebook are self-checks: they catch mistakes as
you work and assign no grade. Treat them as a preview — if a Checkpoint fails
here, the corresponding graded cell will fail there too.

### Where your 40 points come from

| This notebook | Graded part | Points |
|---|---|---:|
| Parts 1–2 — trace the graph, read the shapes off the figure | A — Graph structure invariants | 10 |
| Part 3 — annotate every node with FLOPs and bytes, rank intensity | B — Cost-model self-consistency | 18 |
| Part 4 — classify against the ridge, then change the ridge | C — Ridge classification & the ridge shift | 12 |
| Part 5 — check your understanding | *ungraded self-check* | 0 |
| Part 6 — the Week-2 landscape row | folded into A and B | — |

Part 3 carries the most weight because it is the most work: a per-node cost
model over the whole graph, which everything in Part 4 is then computed from.

### What you write, and what is given to you
The **graph-drawing code in Part 2 is provided**. Laying out a DAG is a plotting
exercise, and this lab is not about `networkx` or Graphviz. Every `TODO` here is
about the dataflow itself: tracing the graph, reading tensor shapes off it,
turning FLOPs and bytes into arithmetic intensity, and classifying nodes against
a *hardware* ridge that you then change.

### Working through this lab
1. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO`. **Part 5 is an ungraded self-check** — nothing there is
   submitted or marked, but it is the reasoning the Concept Check quiz tests.
3. Run **Part 7** to produce **`lab2_results.json`**, then open the graded lab on
   Coursera and paste it in.
4. Take the **Concept Check quiz**, a separate auto-graded Coursera item worth 10 points.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# Setup: install packages that aren't guaranteed to be preinstalled.
# `tabulate` is used by torch.fx's Graph.print_tabular(). It ships with
# Google Colab but not every environment, so install it explicitly.
!pip install -q tabulate

# `pygraphviz` gives networkx the Graphviz `dot` engine, which produces the
# clean layered (top-to-bottom) ranking this notebook's diagrams rely on.
# It needs the Graphviz C library, so we install that first. If either install
# fails the drawing code falls back to a pure-Python layering pass; the figures
# still render, just with slightly less elegant edge routing.
!apt-get -qq install -y graphviz graphviz-dev > /dev/null 2>&1 || true
!pip install -q pygraphviz 2>/dev/null || echo "pygraphviz unavailable; using fallback layout"


In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


## Part 0: Setup

`torch.fx` is PyTorch's mechanism for capturing models as graphs. It works on *symbolic-traceable* modules; roughly, modules whose `forward` doesn't contain data-dependent Python control flow.

Modern LLMs like the full Qwen 2.5 model *are not* cleanly fx-traceable because of KV cache handling, position embedding math, and generation loops. For this lab we use a **simplified but faithful transformer block** that mirrors Qwen's structure (pre-norm, Q/K/V projections, scaled-dot-product attention, SwiGLU-style FFN) while remaining fully traceable.

> In Week 3 you'll see `torch.export` and `torch.compile`'s Dynamo, which *do* handle the full model. FX is the pedagogically cleaner starting point.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.fx
from collections import OrderedDict

torch.manual_seed(0)

print(f"torch = {torch.__version__}")
print(f"device = {'cuda' if torch.cuda.is_available() else 'cpu'}")


> **Roofline & Arithmetic-Intensity Conventions (this notebook)**
> - **Precision:** FP16 (IEEE half) unless otherwise noted
> - **FLOPs counting:** 1 fused multiply-add = 2 FLOPs
> - **Memory level:** Device DRAM (HBM/GDDR), not L2 or scratchpad
> - **Byte convention:** Bytes *transferred* between DRAM and compute (reads + writes)
> - **Sparsity:** Dense (no structural sparsity) unless stated
> - **Hardware SKU:** Detected at runtime via `torch.cuda.get_device_name()`; specs from `hardware_profiles.json`

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


In [ ]:
# Shared hardware reference. identical across every lab in this course.
# ridge = peak FP16 TFLOPS / peak memory bandwidth (TB/s): the arithmetic intensity
# (FLOPs/byte) where a workload flips from memory-bound to compute-bound. These match
# the lecture decks (T4 ridge ~203, H100 ~295).
HW_SPECS = {
    'Tesla T4':              {'fp16_tflops': 65.0,  'bw_tb_s': 0.320, 'ridge': 203},
    'NVIDIA A100-SXM4-40GB': {'fp16_tflops': 312.0, 'bw_tb_s': 1.555, 'ridge': 201},
    'NVIDIA A100-SXM4-80GB': {'fp16_tflops': 312.0, 'bw_tb_s': 2.039, 'ridge': 153},
    'NVIDIA H100':           {'fp16_tflops': 989.0, 'bw_tb_s': 3.35,  'ridge': 295},
}
# IBM Spyre (confirm against the current spec sheet before publishing):
SPYRE_SPECS = {'fp16_tflops': 98.0, 'lpddr5_tb_s': 0.204, 'scratchpad_mb': 64, 'scratchpad_tb_s': 10.0}

def lookup_hw(name):
    for _k, _v in HW_SPECS.items():
        if _k in name:
            return _k, _v
    return 'Tesla T4', HW_SPECS['Tesla T4']   # Colab free-tier default


In [ ]:
class TinyTransformerBlock(nn.Module):
    """
    A traceable transformer block. Dimensions roughly match Qwen 2.5 0.5B:
        hidden_dim = 896, num_heads = 14, head_dim = 64, ffn_dim = 4864.

    For lab tractability we use smaller dims by default. The ratios are preserved
    so the FLOP / memory analysis transfers directly to the real model.

    Architecture: pre-norm, scaled-dot-product attention (no GQA for simplicity),
    SwiGLU-adjacent FFN (gated), residual adds.
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        assert hidden_dim % num_heads == 0
        self.hidden_dim = hidden_dim
        self.num_heads  = num_heads
        self.head_dim   = hidden_dim // num_heads
        self.ffn_dim    = ffn_dim

        # Attention projections
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Norms
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

        # FFN: gated ("SwiGLU-ish"; two up-projections, one down)
        self.ffn_gate = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_up   = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)

    def forward(self, x):
        # x: [batch, seq, hidden]
        B, S, H = x.shape[0], x.shape[1], x.shape[2]
        D = self.head_dim
        N = self.num_heads

        # --- Attention block (pre-norm) ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, N, D).transpose(1, 2)   # [B, N, S, D]
        k = self.k_proj(h).reshape(B, S, N, D).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, N, D).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(D)  # [B, N, S, S]
        attn   = torch.softmax(scores, dim=-1)
        out    = torch.matmul(attn, v)                               # [B, N, S, D]
        out    = out.transpose(1, 2).reshape(B, S, H)
        out    = self.o_proj(out)
        x      = x + out

        # --- FFN block (pre-norm, gated) ---
        h = self.ln2(x)
        gate = torch.nn.functional.silu(self.ffn_gate(h))
        up   = self.ffn_up(h)
        h    = gate * up                                              # gated activation
        h    = self.ffn_down(h)
        x    = x + h
        return x


# Instantiate and smoke-test
HIDDEN_DIM = 512
NUM_HEADS  = 8
FFN_DIM    = 2048
BATCH      = 2
SEQ        = 128

block = TinyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM)
out   = block(dummy)
assert out.shape == dummy.shape
print(f"Block output shape: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")


## Part 1: Trace the block with `torch.fx`

*Feeds **Part A** of the graded notebook (10 points, with Part 2).*

`torch.fx.symbolic_trace` walks through `forward()` with *proxy* tensors, recording every operation that touches a proxy. The output is a `GraphModule` containing a `Graph` of `Node` objects.

Each node has:
- `node.op`; one of `placeholder`, `call_function`, `call_method`, `call_module`, `output`.
- `node.target`: the function/method/module being called.
- `node.args`, `node.kwargs`; its inputs.
- `node.meta`: a dict where you can attach your own metadata (we'll use this in Part 3).

**Your task**:
1. Trace `block` into a `GraphModule`.
2. Print the tabular representation of the graph.
3. Count how many nodes fall into each `op` category.


**💡 Hint: trace + op counts (Part 1)**

<details>
<summary>Click to reveal a hint</summary>

```python
gm = torch.fx.symbolic_trace(block)
gm.graph.print_tabular()

op_counts = {}
for node in gm.graph.nodes:
    op_counts[node.op] = op_counts.get(node.op, 0) + 1
```

</details>

In [ ]:
# TODO 1.1: trace the block with torch.fx.symbolic_trace into `gm`
gm = None  # REPLACE

# TODO 1.2: print the tabular view so you can see the graph structure
# Hint: gm.graph.print_tabular()

# TODO 1.3: count nodes per op type, store in `op_counts` (dict[str, int])
op_counts = {}  # REPLACE

print("\nNode counts by op type:")
for k, v in op_counts.items():
    print(f"  {k:20s}: {v}")
print(f"  total: {sum(op_counts.values())}")


In [ ]:
# --- Checkpoint 1 -------------------------------------------------------
assert isinstance(gm, torch.fx.GraphModule), "gm must be a GraphModule from torch.fx.symbolic_trace"
assert "placeholder" in op_counts and op_counts["placeholder"] >= 1
assert "output" in op_counts and op_counts["output"] == 1
assert "call_module" in op_counts, "Should have call_module nodes (Linears, LayerNorms)"
assert op_counts["call_module"] >= 8, "Expected >= 8 submodule calls (q/k/v/o/ln1/ln2/ffn_gate/ffn_up/ffn_down)"
total = sum(op_counts.values())
assert 20 <= total <= 60, f"Graph size looks off: {total} nodes"
print(f"[OK] Traced block into a {total}-node graph.")


## Part 2: Visualize the graph

*Feeds **Part A** of the graded notebook (10 points, with Part 1). The drawing code is provided; your work is TODO 2.4, reading the dataflow off the figure.*

Reading the tabular print is fine for debugging, but it doesn't give you a feel for
the *dataflow shape*. Let's draw it.

A dataflow graph has a direction: tensors are produced at the top and consumed
below. A force-directed ("spring") layout throws that away and gives you a
hairball, so we lay the graph out in **ranks** instead: every node sits strictly
below the nodes it consumes, the way a compiler's own graph dumps are drawn.
Reading the figure top to bottom then follows the data.

**The drawing code in this Part is provided for you.** Rendering a DAG is a
plotting exercise, not a dataflow one, and this lab is not about learning
`networkx` or Graphviz. Your job is to *read* the figure: the TODO below asks you
to interpret the tensor shapes it puts on the edges.

Three things about the figure are worth knowing before you read it:

- **Every node is a rounded box** labelled with its FX node name and colored by
  `op` type, so `call_module` (the parameterized ops: Linear, LayerNorm) pops.
  Those are the nodes a dataflow compiler spends its effort on.
- **Every edge is labelled with the shape of the tensor flowing along it.** This
  is the part a plain graph hides, and it is what makes the dataflow legible.
- **Scalar plumbing is contracted away.** `x.shape[0]` traces into `getattr` /
  `getitem` nodes that carry integers, not tensors. They are bookkeeping for the
  `reshape` calls, so dropping them keeps the picture about dataflow. (The full
  graph including them is still what Checkpoint 1 counted.)


In [ ]:
# 2.1 (provided): propagate shapes through the graph.
#
# ShapeProp runs one real forward pass and records, for each node, the metadata of
# the tensor it produces -- node.meta["tensor_meta"].shape. The edge labels in the
# figure come from this, and Part 3 reuses the same annotations for FLOPs/bytes.

from torch.fx.passes.shape_prop import ShapeProp

ShapeProp(gm).propagate(dummy)


def shape_of(node):
    """The shape of the tensor this node produces, or None for non-tensor nodes."""
    tm = node.meta.get("tensor_meta")
    if tm is None or not hasattr(tm, "shape"):
        return None
    return list(tm.shape)


_with_shape = sum(1 for n in gm.graph.nodes if shape_of(n) is not None)
print(f"{_with_shape} of {len(list(gm.graph.nodes))} nodes carry a tensor shape.")
print("\nShapes along the attention path:")
for _n in gm.graph.nodes:
    if _n.name in ("x", "ln1", "q_proj", "reshape", "transpose", "matmul",
                   "softmax", "matmul_1", "reshape_3", "o_proj", "add_1"):
        print(f"  {_n.name:12s} -> {shape_of(_n)}")


In [ ]:
# 2.2 (provided): a dataflow-graph renderer for FX graphs.
#
# You do NOT need to read or modify this cell to complete the lab -- just run it.
# It is here so the figure is reproducible and so you can reuse it in later labs.
#
# Two pieces: build_dataflow_graph() turns the FX graph into a networkx DiGraph
# whose *edges* carry the producer's tensor shape, and draw_dag() renders it
# top-to-bottom with rounded node boxes and shape-labelled arrows.
#
# Reused twice: Part 2 colors by op type, Part 4 recolors the same layout by
# roofline class. Computing the layout once and reusing it means the two figures
# are directly comparable -- the same node sits in the same place in both.

import math
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Patch
from matplotlib.path import Path

# How edges are routed between node boxes. Flip this to re-render both figures:
#   "curved" - gentle arcs on sideways edges; vertical edges stay straight.
#   "ortho"  - right-angle elbows (leave the bottom, turn, enter the top), the
#              way hardware dataflow and circuit diagrams are drawn.
#   "line"   - straight centre-to-centre segments.
# Note this is OUR routing, not Graphviz's: `dot` only supplies node positions
# (we read `pos` and discard its own spline geometry), so its `splines` attribute
# has no effect on the picture. This constant is what actually controls it.
EDGE_STYLE = "curved"

# Course palette (shared with every other lab in this course).
TERRACOTTA = "#B85C2C"
CREAM      = "#FAF6F2"
SECONDARY  = "#5C5C5C"
INK        = "#1A1A1A"


def fmt_shape(shape):
    """Compact edge label for a tensor shape: [2,128,512]."""
    if shape is None:
        return ""
    return "[" + ",".join(str(int(d)) for d in shape) + "]"


def build_dataflow_graph(gm, drop_scalar_plumbing=True):
    """FX graph -> networkx DiGraph with the tensor shape attached to each edge.

    An edge u -> v carries the shape of u's output: that is literally the tensor
    handed from u to v. Nodes that produce no tensor (the getattr/getitem pairs
    from `x.shape[0]`) are contracted out by default -- they are shape bookkeeping
    for the reshape calls, not dataflow.
    """
    nodes  = list(gm.graph.nodes)
    shapes = {n.name: shape_of(n) for n in nodes}
    drop   = set()
    if drop_scalar_plumbing:
        drop = {n.name for n in nodes
                if n.op not in ("placeholder", "output") and shapes[n.name] is None}

    G = nx.DiGraph()
    for n in nodes:
        if n.name in drop:
            continue
        G.add_node(n.name, op=n.op, label=n.name, shape=shapes[n.name],
                   cls=n.meta.get("class"), flops=n.meta.get("flops", 0),
                   nbytes=n.meta.get("bytes", 0))
    for n in nodes:
        if n.name in drop:
            continue
        for inp in n.all_input_nodes:
            if inp.name not in drop:
                G.add_edge(inp.name, n.name, shape=shapes[inp.name])
    return G, drop


# --- box geometry ---------------------------------------------------------
# Boxes are sized in data units and the same sizes are handed to Graphviz (in
# inches) so `dot` reserves real space for them and ranks never collide.
BOX_H  = 0.40     # box height, data units
CHAR_W = 0.105    # width per label character
PAD_W  = 0.55     # horizontal padding
UNIT   = 0.62     # data units -> graphviz inches


def box_size(label):
    return max(1.30, CHAR_W * len(label) + PAD_W), BOX_H


def layered_layout(G, rank_gap=0.62, node_sep=1.05):
    """Top-to-bottom layered positions.

    Graphviz `dot` when pygraphviz is importable (best edge routing); otherwise a
    longest-path layering fallback, so this notebook never hard-depends on a
    system library. Either way every node sits strictly below its producers.
    """
    try:
        A = nx.nx_agraph.to_agraph(G)
        # We only consume `dot`'s node positions, not its edge geometry, so no
        # `splines` attribute is set here -- EDGE_STYLE above does the routing.
        A.graph_attr.update(rankdir="TB", nodesep=str(node_sep),
                            ranksep=str(rank_gap))
        for n in G.nodes:
            w, h = box_size(G.nodes[n]["label"])
            A.get_node(n).attr.update(shape="box", fixedsize="true",
                                      width=str(w * UNIT), height=str(h * UNIT))
        A.layout(prog="dot")
        pos = {}
        for n in G.nodes:
            x, y = (float(t) for t in A.get_node(n).attr["pos"].split(","))
            pos[n] = (x / 72.0 / UNIT, y / 72.0 / UNIT)   # points -> data units
        return pos, "graphviz dot"
    except Exception as exc:
        print(f"[layout] Graphviz unavailable ({type(exc).__name__}); "
              f"using the pure-Python fallback layering.")
        return _fallback_layout(G, rank_gap), "fallback layering"


def _fallback_layout(G, rank_gap=0.62):
    """Longest-path layering: rank = longest distance from any source."""
    depth = {}
    for n in nx.topological_sort(G):
        preds = list(G.predecessors(n))
        depth[n] = 0 if not preds else max(depth[p] for p in preds) + 1
    rows = {}
    for n, d in depth.items():
        rows.setdefault(d, []).append(n)
    maxd = max(depth.values()) or 1
    pos  = {}
    for d, members in rows.items():
        for i, n in enumerate(members):
            w, _ = box_size(G.nodes[n]["label"])
            pos[n] = ((i - (len(members) - 1) / 2.0) * (w + 0.6),
                      (maxd - d) * (BOX_H + rank_gap))
    return pos


def _border_point(x, y, w, h, tx, ty):
    """Where the segment (x,y)->(tx,ty) exits the box centred on (x,y).

    Arrows are drawn border-to-border rather than centre-to-centre, so the
    arrowhead lands on the box edge and stays visible.
    """
    dx, dy = tx - x, ty - y
    if dx == 0 and dy == 0:
        return x, y
    s = min((w / 2) / abs(dx) if dx else math.inf,
            (h / 2) / abs(dy) if dy else math.inf)
    return x + dx * s, y + dy * s


def _ink_on(face):
    """Black or white label text, whichever reads on this fill."""
    f = face.lstrip("#")
    r, g, b = (int(f[i:i + 2], 16) for i in (0, 2, 4))
    return "#FFFFFF" if (0.299 * r + 0.587 * g + 0.114 * b) < 140 else INK


def draw_dag(G, pos, color_of, title, legend=None, subtitle=None,
             fontsize=7.5, edge_fontsize=6.2, label_offset=0.16,
             x_scale=0.78, y_scale=0.52, figsize=None, show_shapes=True,
             edge_style=None):
    """Render a dataflow DAG: rounded node boxes, shape-labelled arrows, top-to-bottom.

    color_of(node_name) -> hex fill. legend is an ordered {label: color} dict and
    should contain only classes actually present, so the key never lies.
    edge_style overrides the module-level EDGE_STYLE for a single figure.
    """
    xs = [p[0] for p in pos.values()]
    ys = [p[1] for p in pos.values()]
    if figsize is None:
        figsize = (min(22.0, max(10.0, (max(xs) - min(xs) + 4.0) * x_scale)),
                   min(26.0, max(8.0,  (max(ys) - min(ys) + 2.0) * y_scale)))
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(CREAM)
    fig.patch.set_facecolor(CREAM)

    # Edges first, so the node boxes paint over the arrow tails.
    style = edge_style or EDGE_STYLE
    for u, v, d in G.edges(data=True):
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        w1, h1 = box_size(G.nodes[u]["label"])
        w2, h2 = box_size(G.nodes[v]["label"])

        if style == "ortho":
            # Right-angle elbow: straight down out of the producer, across at the
            # midpoint between the two ranks, then straight down into the consumer.
            sx, sy = x1, y1 - h1 / 2
            ex, ey = x2, y2 + h2 / 2
            if abs(x1 - x2) < 1e-6:
                verts = [(sx, sy), (ex, ey)]          # same column: no turn needed
            else:
                my = (sy + ey) / 2
                verts = [(sx, sy), (sx, my), (ex, my), (ex, ey)]
            ax.add_patch(FancyArrowPatch(
                path=Path(verts, [Path.MOVETO] + [Path.LINETO] * (len(verts) - 1)),
                arrowstyle="-|>", mutation_scale=11, color=SECONDARY, alpha=0.8,
                lw=1.0, shrinkA=0, shrinkB=2, zorder=1, joinstyle="miter"))
            lx, ly = (sx + ex) / 2, (sy + ey) / 2
            nxv, nyv = 0.0, 1.0                       # label sits above the run
        else:
            sx, sy = _border_point(x1, y1, w1, h1, x2, y2)
            ex, ey = _border_point(x2, y2, w2, h2, x1, y1)
            rad = 0.0
            if style == "curved" and abs(x2 - x1) >= 0.35:
                # Bow only the edges that actually travel sideways, so the vertical
                # trunk of the graph stays straight and easy to follow.
                rad = 0.12 if x2 > x1 else -0.12
            ax.add_patch(FancyArrowPatch(
                (sx, sy), (ex, ey), arrowstyle="-|>", mutation_scale=11,
                color=SECONDARY, alpha=0.8, lw=1.0, shrinkA=0, shrinkB=2,
                zorder=1, connectionstyle=f"arc3,rad={rad}"))
            dx, dy = ex - sx, ey - sy
            L = math.hypot(dx, dy) or 1.0
            nxv, nyv = -dy / L, dx / L
            if nyv < 0:                               # always push to the upper side
                nxv, nyv = -nxv, -nyv
            lx, ly = (sx + ex) / 2, (sy + ey) / 2
            if rad:                                   # follow the bow of the arc
                lx += -dy / L * rad * 0.5
                ly += dx / L * rad * 0.5

        if not show_shapes:
            continue
        lbl = fmt_shape(d.get("shape"))
        if not lbl:
            continue
        # Ride the label just off the arrow, along the edge normal.
        ax.text(lx + nxv * label_offset, ly + nyv * label_offset, lbl,
                fontsize=edge_fontsize, color="#3F3F3F", ha="center", va="center",
                zorder=2, bbox=dict(boxstyle="round,pad=0.14", fc=CREAM,
                                    ec="none", alpha=0.9))

    for n, (x, y) in pos.items():
        face = color_of(n)
        w, h = box_size(G.nodes[n]["label"])
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0,rounding_size=0.14",
                                    linewidth=0.9, edgecolor="#2F2F2F",
                                    facecolor=face, zorder=3))
        ax.text(x, y, G.nodes[n]["label"], fontsize=fontsize,
                color=_ink_on(face), ha="center", va="center", zorder=4)

    if subtitle:
        ax.set_title(f"{title}\n", fontsize=13, color=INK, pad=20)
        ax.text(0.5, 1.012, subtitle, transform=ax.transAxes, ha="center",
                va="bottom", fontsize=8.5, color=SECONDARY)
    else:
        ax.set_title(title, fontsize=13, color=INK, pad=16)
    ax.set_xlim(min(xs) - 2.0, max(xs) + 2.0)
    ax.set_ylim(min(ys) - 0.8, max(ys) + 0.8)
    ax.axis("off")
    if legend:
        ax.legend(handles=[Patch(facecolor=c, edgecolor="#2F2F2F", label=k)
                           for k, c in legend.items()],
                  loc="upper left", frameon=False, fontsize=8.5)
    plt.tight_layout()
    return fig, ax


print(f"Renderer ready: build_dataflow_graph(), layered_layout(), draw_dag().")
print(f"Edge routing: EDGE_STYLE = {EDGE_STYLE!r}  (options: 'curved', 'ortho', 'line')")


In [ ]:
# 2.3 (provided): draw the dataflow graph, colored by FX op type.
#
# The layout is computed ONCE here and reused in Part 4, so the two figures are
# directly comparable: a node keeps its position when it is recolored by roofline
# class. Read this figure before answering TODO 2.4 below.

G, dropped = build_dataflow_graph(gm)
POS, LAYOUT_ENGINE = layered_layout(G)

color_for_op = {
    "placeholder":   "#A8C5A8",   # the block input
    "call_module":   TERRACOTTA,  # Linear / LayerNorm -- the parameterized ops
    "call_function": "#5C5C5C",   # matmul, softmax, silu, add, truediv
    "call_method":   "#BFBFBF",   # reshape / transpose -- pure layout changes
    "output":        "#2F2F2F",
    "get_attr":      "#CDCDCD",
}
ops_present = {G.nodes[n]["op"] for n in G.nodes}
legend_ops  = {k: v for k, v in color_for_op.items() if k in ops_present}

fig, ax = draw_dag(
    G, POS,
    color_of=lambda n: color_for_op.get(G.nodes[n]["op"], "#999999"),
    title="TinyTransformerBlock - FX dataflow graph",
    subtitle=f"top to bottom = data direction; edge labels are tensor shapes "
             f"({LAYOUT_ENGINE})",
    legend=legend_ops,
)
plt.show()

n_edges = G.number_of_edges()
print(f"Dataflow graph: {G.number_of_nodes()} nodes, {n_edges} edges.")
print(f"Contracted {len(dropped)} scalar-plumbing node(s): {sorted(dropped)}")
print(f"Full FX graph (what Checkpoint 1 counted): {sum(op_counts.values())} nodes.")


### TODO 2.4: read the dataflow off the figure

The figure above has every tensor shape on it. Three structural events in a
transformer block are visible *purely from how the shapes change along the edges*,
and naming them is the skill this Part is teaching.

Fill in the dictionary below by reading shapes off your figure. Each answer is a
shape tuple you can see on an edge, or a node name you can see on a box.

**💡 What to look for**

<details>
<summary>Click to reveal a hint</summary>

- **The head split.** `q_proj` outputs one flat `[B, S, H]` tensor. Two edges later
  it is `[B, N, S, D]`, with `H = N * D`. Follow `q_proj -> reshape -> transpose`
  and read the three shapes. Nothing was computed; the bytes were only
  *reinterpreted*.
- **The score blowup.** `matmul` (Q·Kᵀ) is the one node whose output is bigger than
  either input. Its shape is quadratic in sequence length: look for the
  `[B, N, S, S]` edge. This is the term that explodes at long context.
- **The FFN widening.** The FFN projects up to `ffn_dim` and back down. Find the
  edge carrying the widest tensor in the whole graph.

</details>


In [ ]:
# TODO 2.4: read these values off the figure you just drew.
#
# This is an interpretation exercise -- every answer is visible on an edge label or
# a node box. Use Python ints/tuples/strings, not prose.

dataflow_reading = {
    # --- the head split: one [B,S,H] tensor reinterpreted as N heads of width D ---
    # The shape ENTERING `reshape` (what q_proj produced):
    "qkv_flat_shape":      None,  # REPLACE with a tuple, e.g. (2, 128, 512)
    # The shape LEAVING `transpose` (what the attention matmul consumes):
    "qkv_head_shape":      None,  # REPLACE with a 4-tuple
    # Does the head split move any data? (does it do FLOPs?)
    "head_split_is_free":  None,  # REPLACE with True or False

    # --- the score blowup: the one output larger than both its inputs ---
    "score_shape":         None,  # REPLACE with the Q.K^T output shape (4-tuple)
    # Which dimension appears TWICE in that shape (the quadratic term)?
    "score_quadratic_dim": None,  # REPLACE with an int (the repeated dim's size)

    # --- the FFN widening ---
    "widest_tensor_shape": None,  # REPLACE with the widest shape in the graph
    # The node that produces the widest tensor FIRST (an FX node name, as a string):
    "widest_producer":     None,  # REPLACE with e.g. "some_node_name"
}

for _k, _v in dataflow_reading.items():
    print(f"  {_k:22s}: {_v}")


In [ ]:
# --- Checkpoint 2 -------------------------------------------------------
# The figure is provided, so this checkpoint verifies your *reading* of it.
assert all(v is not None for v in dataflow_reading.values()), \
    "fill in every entry of dataflow_reading"

_edge_shapes = {tuple(d["shape"]) for _, _, d in G.edges(data=True)}

# Head split: flat [B,S,H] -> per-head [B,N,S,D], and H must equal N*D.
_flat = tuple(dataflow_reading["qkv_flat_shape"])
_head = tuple(dataflow_reading["qkv_head_shape"])
assert _flat in _edge_shapes, f"{_flat} is not a shape on any edge in your graph"
assert _head in _edge_shapes, f"{_head} is not a shape on any edge in your graph"
assert len(_flat) == 3 and len(_head) == 4, "flat is 3-D [B,S,H]; per-head is 4-D"
assert _flat[2] == _head[1] * _head[3], \
    f"hidden {_flat[2]} should equal num_heads * head_dim = {_head[1]}*{_head[3]}"
assert dataflow_reading["head_split_is_free"] is True, \
    "reshape/transpose only reinterpret layout -- no FLOPs, so the split is free here"

# Score blowup: [B,N,S,S], quadratic in sequence length.
_score = tuple(dataflow_reading["score_shape"])
assert _score in _edge_shapes, f"{_score} is not a shape on any edge"
assert len(_score) == 4 and _score[2] == _score[3], \
    "the Q.K^T output should be [B, N, S, S] -- the last two dims equal"
assert dataflow_reading["score_quadratic_dim"] == _score[3], \
    "the quadratic dimension is the one appearing twice (sequence length)"

# FFN widening: the widest tensor anywhere in the graph.
_widest = tuple(dataflow_reading["widest_tensor_shape"])
def _numel(s):
    p = 1
    for d in s: p *= d
    return p
assert _widest in _edge_shapes, f"{_widest} is not a shape on any edge"
assert _numel(_widest) == max(_numel(s) for s in _edge_shapes), \
    "there is a larger tensor on some edge -- find the widest one"
_prod = dataflow_reading["widest_producer"]
assert _prod in G, f"'{_prod}' is not a node in the graph"
assert tuple(G.nodes[_prod]["shape"]) == _widest, \
    f"'{_prod}' does not produce {_widest}"

print(f"[OK] Dataflow read correctly: head split {_flat} -> {_head}, "
      f"scores {_score}, widest {_widest} from '{_prod}'.")


## Part 3: Annotate every node with compute cost and memory footprint

*Feeds **Part B** of the graded notebook — **18 points, the largest single block in the lab**. Everything in Part 4 is computed from the numbers you produce here.*

A dataflow compiler needs two numbers for every node to make scheduling and tiling decisions:
- **FLOPs**: how much compute does this node do?
- **Bytes moved**: how much data must flow across the inputs and output?

Their ratio; *arithmetic intensity* (AI); is the most important single number in performance analysis. `AI = FLOPs / bytes`. A node with AI > 10 is likely compute-bound; AI < 1 is memory-bound.

We'll do a simple first-order model:

| Op | FLOPs | Bytes (fp16) |
|---|---|---|
| `nn.Linear(d_in, d_out)` on `[B, S, d_in]` | `2 * B * S * d_in * d_out` | `2 * (B*S*d_in + d_in*d_out + B*S*d_out)` |
| `matmul([B,N,S,D] @ [B,N,D,S])` (Q·Kᵀ) | `2 * B * N * S * S * D` | `2 * (B*N*S*D + B*N*D*S + B*N*S*S)` |
| `matmul([B,N,S,S] @ [B,N,S,D])` (attn·V) | `2 * B * N * S * S * D` | `2 * (B*N*S*S + B*N*S*D + B*N*S*D)` |
| `LayerNorm`, `softmax`, `silu`, `mul`, `add` | ~0 FLOPs (relative), bytes = 2 × elementwise tensor size | (memory-bound) |

**Your task**: for each node in `gm.graph`, attach `node.meta["flops"]` and `node.meta["bytes"]` using the helper functions provided.


**💡 Hint: annotate FLOPs & bytes (Part 3.1)**

<details>
<summary>Click to reveal a hint</summary>

- `ShapeProp(gm).propagate(dummy)` (already in the cell) gives every node a `node.meta["tensor_meta"].shape`.
- `placeholder / output / get_attr` → leave `flops=0, bytes=0`.
- `call_module` → `sub = gm.get_submodule(node.target)`. If `isinstance(sub, nn.Linear)`, use `linear_flops/linear_bytes` with `batch_seq` = product of the input shape except its last dim. If `nn.LayerNorm`, use `elementwise_bytes(out_shape)`.
- `call_function` with `node.target is torch.matmul` → `matmul_flops(a, b)` / `matmul_bytes(a, b, out)` from the two input shapes.
- Everything else (softmax, silu, mul, add, transpose, reshape, truediv) → `bytes = elementwise_bytes(out_shape)`, `flops = 0`.

</details>

In [ ]:
# Helpers. these compute FLOPs and bytes given input/output shapes.
# You use them below; you don't need to modify them.

BYTES_PER_ELEM = 2  # fp16 assumption for the analysis

def linear_flops(in_features, out_features, batch_seq):
    # 2 * B * S * in_features * out_features (multiply + add per output elem)
    return 2 * batch_seq * in_features * out_features

def linear_bytes(in_features, out_features, batch_seq):
    # Input + weights + output, in fp16.
    elems = batch_seq * in_features + in_features * out_features + batch_seq * out_features
    return BYTES_PER_ELEM * elems

def matmul_flops(a_shape, b_shape):
    # a: (..., M, K), b: (..., K, N). FLOPs = 2 * M * N * K * batch_dims_product.
    bat = 1
    for d in a_shape[:-2]:
        bat *= d
    M, K = a_shape[-2], a_shape[-1]
    N    = b_shape[-1]
    return 2 * bat * M * N * K

def matmul_bytes(a_shape, b_shape, out_shape):
    def prod(s):
        p = 1
        for d in s: p *= d
        return p
    return BYTES_PER_ELEM * (prod(a_shape) + prod(b_shape) + prod(out_shape))

def elementwise_bytes(shape):
    # Read one tensor, write one tensor.
    p = 1
    for d in shape: p *= d
    return BYTES_PER_ELEM * 2 * p

# ShapeProp already ran in Part 2.1, so every node already has
# node.meta["tensor_meta"] with .shape and .dtype. shape_of() is defined there too.

# TODO 3.1: walk every node. For each, compute FLOPs and bytes and attach
#             to node.meta["flops"] and node.meta["bytes"].
#
#             Follow these rules:
#               * placeholder / output / get_attr  -> flops=0, bytes=0
#               * call_module to nn.Linear         -> linear_flops / linear_bytes
#                   (use `gm.get_submodule(node.target)` to get the Linear;
#                    its in/out shape come from the input arg's shape + Linear.out_features)
#               * call_module to nn.LayerNorm      -> flops=0 (small), bytes = elementwise_bytes(output)
#               * call_function torch.matmul       -> matmul_flops / matmul_bytes
#                   (inspect args: both are FX nodes, with shapes in .meta["tensor_meta"].shape)
#               * everything else (softmax, silu, mul, add, transpose, reshape, truediv)
#                   -> flops=0, bytes = elementwise_bytes(output shape)
#
# The solution fills in roughly 25 lines. If yours is much longer, simplify.

for node in gm.graph.nodes:
    node.meta["flops"] = 0
    node.meta["bytes"] = 0
    # YOUR CODE HERE

# Print the annotated table
print(f"{'name':30s} {'op':15s} {'flops':>14s} {'bytes':>14s} {'AI':>8s}")
print("-" * 85)
for node in gm.graph.nodes:
    f = node.meta["flops"]
    b = node.meta["bytes"]
    ai = (f / b) if b > 0 else 0.0
    print(f"{node.name:30s} {node.op:15s} {f:14,d} {b:14,d} {ai:8.2f}")


In [ ]:
# --- Checkpoint 3 ------------------------------------------------------
# Every node should have flops and bytes keys set.
assert all("flops" in n.meta and "bytes" in n.meta for n in gm.graph.nodes)

# Total FLOPs should be dominated by the matmuls and Linears.
total_flops = sum(n.meta["flops"] for n in gm.graph.nodes)
total_bytes = sum(n.meta["bytes"] for n in gm.graph.nodes)

# Rough expected FLOP count for this block at HIDDEN=512 NUM_HEADS=8 FFN=2048 B=2 S=128
# (batch_seq = B*S = 256):
#   4 QKVO Linears : 4 * (2 * 256 * 512 * 512)                ~ 5.37e8
#   2 attn matmuls : 2 * (2 * 2 * 8 * 128 * 128 * 64)         ~ 6.71e7
#   3 FFN Linears  : (gate) 2*256*512*2048 + (up) 2*256*512*2048
#                    + (down) 2*256*2048*512                  ~ 1.61e9
#   total                                                     ~ 2.2e9
assert 5e8 < total_flops < 5e9, f"Total FLOPs look off: {total_flops:,}"
assert total_bytes > 0
print(f"[OK] Total FLOPs: {total_flops:,}  |  Total bytes: {total_bytes:,}")


### TODO 3.2: arithmetic intensity, and what it means per node

`AI = FLOPs / bytes` is the single most important number in this course. You have
FLOPs and bytes on every node; now turn them into intensity and rank them.

Two things to notice as you do it, because both come back in Weeks 4 and 6:

- The `reshape` / `transpose` nodes have **AI = 0**. They move bytes and compute
  nothing. On a GPU they are nearly free (a stride change). On a dataflow target
  with fixed on-chip layouts they may require *physical reshuffling*.
- The three FFN Linears have the **highest** AI in the block, and the attention
  matmuls are much lower; `matmul` re-reads Q and K to produce an `[B,N,S,S]`
  tensor that is pure intermediate traffic.


In [ ]:
# TODO 3.2: compute arithmetic intensity per node and rank the top consumers.

# 3.2a: attach node.meta["ai"] = flops / bytes for every node (0.0 when bytes == 0).
for node in gm.graph.nodes:
    node.meta["ai"] = None  # REPLACE: flops / bytes, guarding against bytes == 0

# 3.2b: the five nodes with the highest arithmetic intensity, highest first.
#        Each entry is (node_name, ai). Only include nodes with bytes > 0.
top5_ai = None  # REPLACE with a list of 5 (name, ai) tuples

# 3.2c: every node that does real data movement but ZERO compute (bytes > 0,
#        flops == 0) -- the pure-memory-traffic ops a compiler wants to fuse away.
zero_flop_movers = None  # REPLACE with a list of node names (strings)

print(f"{'name':24s} {'flops':>14s} {'bytes':>12s} {'AI':>9s}")
print("-" * 64)
for node in gm.graph.nodes:
    print(f"{node.name:24s} {node.meta['flops']:14,d} "
          f"{node.meta['bytes']:12,d} {node.meta['ai']:9.2f}")

print("\nTop 5 by arithmetic intensity:")
for _name, _ai in (top5_ai or []):
    print(f"  {_name:24s} AI = {_ai:8.2f}")
print(f"\nZero-FLOP data movers ({len(zero_flop_movers or [])}): "
      f"{sorted(zero_flop_movers or [])}")


In [ ]:
# --- Checkpoint 4 -----------------------------------------------------
assert all(n.meta.get("ai") is not None for n in gm.graph.nodes), \
    "every node needs node.meta['ai']"

# AI must be consistent with the flops/bytes you attached in 3.1.
for n in gm.graph.nodes:
    _want = (n.meta["flops"] / n.meta["bytes"]) if n.meta["bytes"] > 0 else 0.0
    assert abs(n.meta["ai"] - _want) < 1e-6, f"AI wrong for {n.name}"

assert top5_ai is not None and len(top5_ai) == 5, "top5_ai must hold 5 entries"
_ais = [a for _, a in top5_ai]
assert _ais == sorted(_ais, reverse=True), "top5_ai must be sorted highest-first"
# The FFN Linears dominate intensity in this block.
_top_names = {nm for nm, _ in top5_ai}
assert {"ffn_gate", "ffn_up", "ffn_down"} <= _top_names, \
    f"expected the three FFN Linears among the top 5; got {sorted(_top_names)}"

assert zero_flop_movers is not None, "fill in zero_flop_movers"
_zfm = set(zero_flop_movers)
_want_zfm = {n.name for n in gm.graph.nodes
             if n.meta["bytes"] > 0 and n.meta["flops"] == 0}
assert _zfm == _want_zfm, (
    f"missing {sorted(_want_zfm - _zfm)}; unexpected {sorted(_zfm - _want_zfm)}")
# These are exactly the reshape/transpose/softmax/elementwise ops.
assert any(nm.startswith("transpose") for nm in _zfm), \
    "the transposes move bytes and compute nothing -- they belong here"
print(f"[OK] AI computed. Top node: {top5_ai[0][0]} at AI={top5_ai[0][1]:.1f}; "
      f"{len(_zfm)} zero-FLOP data movers.")


## Part 4 - Classify each node

*Feeds **Part C** of the graded notebook (12 points).*

The arithmetic-intensity **ridge** -- where a workload flips from memory-bound to compute-bound -- is `peak_compute / peak_bandwidth`, your device's real ridge (about **203 FLOPs/byte on a T4**, **295 on an H100**). We classify each node against it:

- **compute-bound**  if `AI >= ridge`
- **memory-bound**   if `0 < AI < ridge`
- **cheap**          if `bytes < 1e4` (tiny tensors -- noise)

At this toy block size you will find most GEMMs land **below** the ridge -- they are memory-bound even though they are matmuls. That is exactly the Week 6 lesson: the ridge is high, so you have to be huge to be compute-bound. Later weeks revisit the threshold for Spyre (scratchpad shifts the ridge) and TPU (VMEM + HBM give two ridges).

The figure below recolors **the same layout** you built in Part 2: every node sits
where it sat before, so you can flip between the two pictures and read the
classification straight onto the dataflow shape. The tensor shapes stay on the
edges, which is the point -- you can see *why* a node is memory-bound by looking
at what flows into it.

The figure below recolors **the same layout** you read in Part 2: every node sits
where it sat before, so you can flip between the two pictures and read the
classification straight onto the dataflow shape. The tensor shapes stay on the
edges, which is the point: you can see *why* a node is memory-bound by looking at
what flows into it.


In [ ]:
_hwname, _hw = lookup_hw(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tesla T4")
GPU_RIDGE_AI = _hw["ridge"]
print(f"Classifying against {_hwname} ridge = {GPU_RIDGE_AI} FLOPs/byte")
CHEAP_BYTES  = 1e4

# TODO 4.1: classify every node. Attach node.meta["class"] in {"compute", "memory", "cheap"}.
# Then compute counts per class into `class_counts`.
class_counts = {"compute": 0, "memory": 0, "cheap": 0}
# YOUR CODE HERE

print("Node classification:")
for k, v in class_counts.items():
    print(f"  {k:10s}: {v}")


In [ ]:
# 4.2 (provided): recolor the same dataflow graph by roofline class.
#
# Same nodes, same positions, same shape-labelled edges as Part 2 -- only the fill
# changes. Reusing POS is deliberate: a node that was terracotta-because-Linear in
# Part 2 may be gray-because-memory-bound here, and holding the layout fixed makes
# that jump legible.

# Part 2 built G before you assigned node.meta["class"], so refresh the drawing's
# copy of the per-node analysis from the FX graph.
for _n in gm.graph.nodes:
    if _n.name in G:
        G.nodes[_n.name]["cls"]    = _n.meta.get("class")
        G.nodes[_n.name]["flops"]  = _n.meta.get("flops", 0)
        G.nodes[_n.name]["nbytes"] = _n.meta.get("bytes", 0)

class_color = {
    "compute": TERRACOTTA,   # AI >= ridge: the heavy lifting
    "memory":  "#5C5C5C",    # 0 < AI < ridge: starved by bandwidth
    "cheap":   "#CDC5BA",    # tiny tensors; below the noise floor
}
classes_present = {G.nodes[n]["cls"] for n in G.nodes}
legend_cls = {k: v for k, v in class_color.items() if k in classes_present}

fig, ax = draw_dag(
    G, POS,
    color_of=lambda n: class_color.get(G.nodes[n]["cls"], "#999999"),
    title="Roofline classification on the dataflow graph",
    subtitle=f"{_hwname} ridge = {GPU_RIDGE_AI} FLOPs/byte   |   "
             f"terracotta = compute-bound, gray = memory-bound, sand = cheap",
    legend=legend_cls,
)
plt.show()

print("Classification on the drawn graph:")
for _cls in ("compute", "memory", "cheap"):
    _members = [n for n in G.nodes if G.nodes[n]["cls"] == _cls]
    print(f"  {_cls:8s}: {len(_members):2d}  {sorted(_members)}")


In [ ]:
# --- Checkpoint 5 ------------------------------------------------------
assert sum(class_counts.values()) == sum(op_counts.values()), \
    "Every node must be classified exactly once."
# Sanity: you should have at least a few compute-bound nodes (the Linears, matmuls).
# At the true device ridge (~203 on a T4) most ops in this toy block are memory-bound --
# the Week 6 punchline arriving early. Require only a valid partition + memory-bound nodes.
assert class_counts["memory"] >= 2, f"Expected memory-bound nodes; got {class_counts}"
# ... and some memory-bound / cheap ones (softmax, elementwise mul, add, etc).
assert class_counts["compute"] + class_counts["memory"] + class_counts["cheap"] == sum(op_counts.values())
print(f"[OK] Classification complete: {class_counts}")


### TODO 4.3: the ridge is a property of the *hardware*, not the model

You just classified every node against the T4's ridge (~203 FLOPs/byte). The
model did not change. Now change the hardware and watch the answer move.

This is the central idea behind dataflow accelerators. A GPU streams operands from
HBM, so its ridge is set by HBM bandwidth. A dataflow accelerator like **Spyre**
keeps tensors in a large on-chip **scratchpad** and streams between operators, so
the relevant bandwidth is scratchpad bandwidth, which is far higher. A higher
bandwidth for the same compute means a **lower** ridge, and a lower ridge means
**more** of your graph counts as compute-bound.

`SPYRE_SPECS` is already defined in Part 0. Compute both ridges and re-classify.


In [ ]:
# TODO 4.3: compute the ridge for two memory systems and re-classify.
#
# ridge = peak_compute (FLOP/s) / peak_bandwidth (byte/s)
# With TFLOPS and TB/s the units cancel, so: ridge = fp16_tflops / bw_tb_s.

# 4.3a: Spyre's ridge if operands stream from off-chip LPDDR5.
spyre_ridge_lpddr = None  # REPLACE: SPYRE_SPECS fp16_tflops / lpddr5_tb_s

# 4.3b: Spyre's ridge if operands stay resident in the on-chip scratchpad.
spyre_ridge_scratchpad = None  # REPLACE: fp16_tflops / scratchpad_tb_s

# 4.3c: re-classify every node against the scratchpad ridge. Same rule as 4.1
#        (cheap if bytes < CHEAP_BYTES, else compute if AI >= ridge, else memory),
#        but counted into a fresh dict -- do NOT overwrite node.meta["class"].
spyre_class_counts = {"compute": 0, "memory": 0, "cheap": 0}
# YOUR CODE HERE

print(f"T4 ridge (HBM)              : {GPU_RIDGE_AI:8.1f} FLOPs/byte")
print(f"Spyre ridge (LPDDR5)        : {spyre_ridge_lpddr:8.1f} FLOPs/byte")
print(f"Spyre ridge (scratchpad)    : {spyre_ridge_scratchpad:8.1f} FLOPs/byte")
print()
print(f"Nodes compute-bound on T4 (ridge {GPU_RIDGE_AI:.0f}):"
      f" {class_counts['compute']}")
print(f"Nodes compute-bound on Spyre scratchpad "
      f"(ridge {spyre_ridge_scratchpad:.1f}): {spyre_class_counts['compute']}")


In [ ]:
# --- Checkpoint 6 -----------------------------------------------------
assert spyre_ridge_lpddr is not None and spyre_ridge_scratchpad is not None, \
    "compute both Spyre ridges"

_want_lpddr = SPYRE_SPECS["fp16_tflops"] / SPYRE_SPECS["lpddr5_tb_s"]
_want_spad  = SPYRE_SPECS["fp16_tflops"] / SPYRE_SPECS["scratchpad_tb_s"]
assert abs(spyre_ridge_lpddr - _want_lpddr) < 1.0, \
    f"LPDDR5 ridge should be ~{_want_lpddr:.1f}"
assert abs(spyre_ridge_scratchpad - _want_spad) < 0.5, \
    f"scratchpad ridge should be ~{_want_spad:.1f}"

# The scratchpad is ~49x the bandwidth of LPDDR5, so its ridge is far lower.
assert spyre_ridge_scratchpad < spyre_ridge_lpddr, \
    "more bandwidth for the same compute means a LOWER ridge"

assert sum(spyre_class_counts.values()) == sum(op_counts.values()), \
    "every node must be classified exactly once against the scratchpad ridge"

# The punchline: a lower ridge promotes memory-bound nodes to compute-bound.
assert spyre_class_counts["compute"] >= class_counts["compute"], \
    "a lower ridge cannot reduce the compute-bound count"
assert spyre_class_counts["compute"] > 0, (
    "at the scratchpad ridge the GEMMs should clear the bar -- check your "
    "re-classification")
# node.meta["class"] must still hold the T4 classification.
assert all(n.meta["class"] in ("compute", "memory", "cheap") for n in gm.graph.nodes)
print(f"[OK] Ridge shift: T4 {GPU_RIDGE_AI} -> Spyre scratchpad "
      f"{spyre_ridge_scratchpad:.1f} FLOPs/byte moved "
      f"{spyre_class_counts['compute'] - class_counts['compute']} node(s) "
      f"into compute-bound.")


## Part 5: Check your understanding (ungraded)

These prompts are **not graded and not submitted** — nothing here travels in your
JSON. Work through them anyway: they are the reasoning the **Concept Check quiz**
tests, and that quiz *is* worth 10 points. Answer in your own words, citing node
names from *your own* trace, then compare against the solution notebook once it
unlocks.

A **dataflow accelerator** executes the graph you just built: each node fires
when its inputs are ready. Three things tend to become bottlenecks on dataflow
targets that *don't* hurt as much on a GPU:

1. **Shape transitions** — `transpose`, `reshape`, `permute` are nearly free on a
   GPU (a stride change, no data movement). On a dataflow target with fixed
   on-chip layouts they can force *physical* data reshuffling.
2. **Reductions along the wrong axis** — a softmax across the SRAM-tiled axis
   stalls the pipeline waiting for partial sums.
3. **Small ops between big ones** — GPUs amortize them through caches; a dataflow
   pipeline may have to drain.

---

### 5.1: Which three nodes in *your* graph would become scheduling bottlenecks on a dataflow target?

Pick three specific node names from your own trace. For each, say which of the
three categories above applies and why. Your Part 3 table already tells you which
nodes move bytes without doing FLOPs — use it.

*Your reasoning here (ungraded — for your own learning).*

---

### 5.2: Find one fusion opportunity in your graph.

Identify a concrete chain of nodes where fusion looks obviously beneficial, and
describe what the single fused node would read once, hold in scratchpad, and write
back. Then quantify it from your own per-node bytes: roughly how much memory
traffic does fusing that chain remove? State the before and after.

Hint: look at the chain `ln2 → ffn_gate → silu → mul → ffn_down`, and at which of
its members appear in your `zero_flop_movers` list.

*Your reasoning here (ungraded — for your own learning).*

---

### 5.3: What happens to the arithmetic intensity of Q·Kᵀ at sequence length 4096?

Show the calculation. Q and K are `[B, N, S, D]` and the score tensor is
`[B, N, S, S]`; write FLOPs and bytes as functions of `S`, then take the ratio.

Then answer: does the matmul move *toward* compute-bound or *toward*
memory-bound — and is that the same thing as its absolute memory pressure going
up or down? Those two questions have different answers, and distinguishing them
is the point. Week 4's KV-cache discussion depends on it.

*Your reasoning here (ungraded — for your own learning).*


## Part 6: Update the Landscape Map

*Checked in Parts A and B of the graded notebook.*

Week 2 adds a new row to the shared Accelerator Landscape Map: **"Graph representation complexity"**: how much work does each system's compiler do to get from the source PyTorch graph to something executable?

Fill in the row for PyTorch-on-GPU based on what you observed. We will fill in other accelerators in Weeks 3–4.


In [ ]:
# TODO 6.1: add the Week-2 Landscape Map row.
# Build a clean accelerator label first, so a CPU runtime doesn't show up as "GPU (CPU)":
#     accelerator_name = (
#         f"GPU ({torch.cuda.get_device_name(0)})"
#         if torch.cuda.is_available()
#         else "CPU runtime"
#     )
landscape_row_w2 = {
    "accelerator":              None,  # use accelerator_name from the hint above
    "graph_capture_mechanism":  None,  # e.g. "torch.fx symbolic_trace (static) / TorchDynamo (production)"
    "graph_size":               None,  # e.g. f"{G.number_of_nodes()} nodes / {n_edges} edges"
    "compile_time_work":        None,  # NOTE: this lab only does FX graph *capture*; it does NOT compile,
                                       #       fuse, or generate kernels. e.g. "FX graph capture only in this
                                       #       lab; with torch.compile, TorchInductor may perform fusion and
                                       #       kernel code generation"
    "scheduling_granularity":   None,  # e.g. "kernel-level once compiled (operators fused into CUDA kernels)"
    "compute_bound_nodes":      None,  # class_counts["compute"]
    "memory_bound_nodes":       None,  # class_counts["memory"]
}

for k, v in landscape_row_w2.items():
    print(f"  {k:28s}: {v}")

In [ ]:
# --- Checkpoint 7 ------------------------------------------------------
required = {"accelerator", "graph_capture_mechanism", "graph_size",
            "compile_time_work", "scheduling_granularity",
            "compute_bound_nodes", "memory_bound_nodes"}
assert set(landscape_row_w2.keys()) == required
assert all(v is not None for v in landscape_row_w2.values())
print("[OK] Week 2 Landscape Map row complete.")


---
## Part 7: Save and submit your results

Run the cell below. It writes **`lab2_results.json`** and prints it inline so you
can copy it without opening the file browser.

Then open the graded notebook on Coursera Labs, paste the JSON into its
`RESULTS_JSON` cell, and run every cell top to bottom. **That notebook is what is
graded — 40 points, all autograded.** It tells you your score before you submit.

Everything it checks comes from values you computed in Parts 1-6: the FX graph's
op histogram and node/edge counts, the shapes you read off the figure, the **raw
per-node FLOPs and bytes**, the arithmetic-intensity ranking, the zero-FLOP data
movers, both class partitions, and your Week-2 landscape row. The grader
recomputes every derived value from your raw per-node numbers, so that table is
the part that matters.

> **Nothing from Part 5 is submitted.** It is an ungraded self-check — the
> concepts it covers are tested by the separate **Concept Check quiz** (10 points).


In [ ]:
# === Part 7: Save and submit your results ==============================
# This cell writes lab2_results.json AND prints it inline, so you can copy it
# straight from the output without opening the file browser.
#
# Everything the graded notebook needs is here. It exports the RAW per-node
# FLOPs and bytes alongside the derived totals and class counts, so the grader
# can recompute every derived value from your own raw numbers.
import json as _json

# Per-node raw measurements: the ground truth every derived value comes from.
_nodes = [
    {
        "name":  n.name,
        "op":    n.op,
        "flops": int(n.meta["flops"]),
        "bytes": int(n.meta["bytes"]),
        "ai":    float(n.meta["ai"]),
        "class": n.meta["class"],
    }
    for n in gm.graph.nodes
]

my_results = {
    # --- Part 1: FX graph structure -----------------------------------
    "op_counts":        op_counts,
    "n_fx_nodes":       sum(op_counts.values()),
    # --- Part 2: the drawn dataflow graph + what you read off it ------
    "n_nodes":          G.number_of_nodes(),
    "n_edges":          n_edges,
    "dataflow_reading": {k: (list(v) if isinstance(v, tuple) else v)
                         for k, v in dataflow_reading.items()},
    # --- Part 3: per-node cost model (RAW) ----------------------------
    "nodes":            _nodes,
    "total_flops":      sum(n.meta["flops"] for n in gm.graph.nodes),
    "total_bytes":      sum(n.meta["bytes"] for n in gm.graph.nodes),
    "top5_ai":          [[nm, float(a)] for nm, a in top5_ai],
    "zero_flop_movers": sorted(zero_flop_movers),
    # --- Part 4: classification, and the ridge shift ------------------
    "cheap_bytes":            CHEAP_BYTES,
    "gpu_ridge_ai":           GPU_RIDGE_AI,
    "class_counts":           class_counts,
    "spyre_ridge_lpddr":      spyre_ridge_lpddr,
    "spyre_ridge_scratchpad": spyre_ridge_scratchpad,
    "spyre_class_counts":     spyre_class_counts,
    # --- Part 6: landscape map row ------------------------------------
    "landscape_row_w2": landscape_row_w2,
}

with open("lab2_results.json", "w") as _f:
    _json.dump(my_results, _f, indent=2, default=str)

print("=" * 70)
print("lab2_results.json  --  copy everything between the lines below")
print("=" * 70)
print(_json.dumps(my_results, indent=2, default=str))
print("=" * 70)

print("""
NEXT STEPS
----------
1. Select and copy the JSON printed above (from the opening '{' through the
   final '}').  If it is easier, open the file browser (folder icon, left
   sidebar), double-click lab2_results.json, select all and copy.
2. Open the graded notebook on Coursera Labs.
3. Paste it into the RESULTS_JSON cell there, then run every cell top to
   bottom.  That notebook is what is graded -- 40 points, all autograded.
   It prints a score table so you know your total before you submit.
4. The Concept Check quiz is a SEPARATE auto-graded Coursera item worth
   10 points.  Take it as well: 40 + 10 = the lab's 50 points.
5. The solution notebook unlocks on Coursera once you pass the graded lab.

Keep this Colab notebook: if a graded check fails you will want to re-run the
cell that produced the value.
""")

# Escape hatch: uncomment to download the file instead of copying the text.
# from google.colab import files
# files.download("lab2_results.json")

---

## Bonus. Graph Breaks: What Happens When Tracing Fails (Preview of Week 5)

The `TinyTransformerBlock` above was designed to be fully traceable. But what happens when
a model contains operations that `torch.fx` **cannot** capture? The result is a "graph break"; the graph is split into fragments, and the untraceable code falls back to eager Python.

Run the cell below to see what a graph break looks like. This is a preview of Week 5's
debugging content; you will learn to detect and fix these systematically.

In [ ]:
import torch
import torch.nn as nn
import torch.fx

class BrokenBlock(nn.Module):
    """A module with a graph-breaking .item() call."""
    def __init__(self, d_model=64):
        super().__init__()
        self.linear = nn.Linear(d_model, d_model)
        self.threshold = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        out = self.linear(x)
        # .item() forces a Python scalar; symbolic tracing cannot handle this
        if self.threshold.item() > 0.3:
            out = torch.relu(out)
        return out

model = BrokenBlock()
x = torch.randn(1, 10, 64)

# Attempt symbolic trace. This will FAIL
try:
    traced = torch.fx.symbolic_trace(model)
    print("Trace succeeded (unexpected!)")
    traced.graph.print_tabular()
except Exception as e:
    print(f"Graph break! torch.fx.symbolic_trace failed:")
    print(f"   {type(e).__name__}: {e}")
    print()
    print("Why? .item() requires a concrete value at trace time,")
    print("but symbolic tracing only passes proxy objects.")
    print()
    print("In Week 5, you'll learn how torch.compile (TorchDynamo)")
    print("handles this via graph breaks; splitting the graph into")
    print("fragments and falling back to eager Python for the .item() section.")

---

## Optional: Dataflow Deep Dive (Spyre hardware access required)

> *Only complete this section if you have been granted Spyre access through the IBM Research academic partnership.*

Compile this same block with the Spyre compiler, dump its internal dataflow IR,
and catalog what the compiler **changed** from your FX graph (fused ops,
reordered nodes, inserted data-movement edges). You can then compare its fusion
decisions against your own reasoning in Part 5.2.

This is enrichment only: it earns **no extra credit**, and the lab's 50 points
are fully reachable from the free Colab tier.

---

*Course: Dataflow Architectures for AI, Week 2 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*


> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.
